# Week 9: Raster & Remote Sensing

This notebook covers:
- Loading satellite imagery with Rasterio
- Clipping rasters to an area of interest
- **Calculating NDVI (Normalized Difference Vegetation Index)**
- Detecting vegetation change between two dates
- Zonal statistics

---

## Before you start

You'll need satellite imagery files for this notebook. Make sure you've:
1. Downloaded the Week 9 data from the course data guide
2. Uploaded it to your Google Drive (Colab) or saved it locally (Jupyter)

**Important:** Your satellite imagery needs multiple bands (Red and Near-Infrared at minimum) for NDVI calculation.

---

## Step 0: Set up environment

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages...")
    !pip install geopandas rasterio rasterstats -q
    print("Done!")
else:
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")

---

## Step 1: Connect to your data

**Colab users:** This mounts your Google Drive so the notebook can access your files.

Your Drive folder should look like:
```
My Drive/
└── intro-gis/
    └── week09/
        └── data/
            ├── raw/              ← Input data goes here
            │   ├── aoi.geojson
            │   ├── sentinel_before.tif
            │   ├── sentinel_after.tif
            │   └── zones.geojson
            └── processed/        ← Your outputs save here
```

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW = Path("/content/drive/MyDrive/intro-gis/week09/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week09/data/processed")
else:
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

print(f"Reading input data from: {RAW}")
print(f"Saving outputs to: {PROCESSED}")

if RAW.exists():
    print("\nInput folder found! Files:")
    for f in RAW.glob("*"):
        print(f"  {f.name}")
else:
    print("\nInput folder NOT found - check your Drive/local folder structure")

# Create processed folder if it doesn't exist
PROCESSED.mkdir(parents=True, exist_ok=True)
print(f"\nProcessed folder ready: {PROCESSED}")

---

## Step 2: Import libraries

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported!")

---

## Step 3: Load area of interest

The AOI defines the boundary we'll clip our satellite imagery to.

In [ ]:
aoi = gpd.read_file(RAW / "aoi.geojson")

print(f"AOI bounds: {aoi.total_bounds}")
aoi.plot(figsize=(8, 6))
plt.title("Area of Interest")
plt.show()

---

## Step 4: Clip rasters to AOI

Crop the satellite images to your study area (like Clip Raster by Mask Layer in QGIS).

In [ ]:
def clip_raster(raster_path, shapes):
    """Clip a raster to a vector boundary."""
    with rasterio.open(raster_path) as src:
        out_image, out_transform = mask(src, shapes.geometry, crop=True)
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
        })
    return out_image, out_meta

# Clip both images
before, before_meta = clip_raster(RAW / "sentinel_before.tif", aoi)
after, after_meta = clip_raster(RAW / "sentinel_after.tif", aoi)

print(f"Clipped raster shape: {before.shape}")
print(f"  Bands: {before.shape[0]}, Height: {before.shape[1]}, Width: {before.shape[2]}")

---

## Step 5: Understanding NDVI

### What is NDVI?

**NDVI (Normalized Difference Vegetation Index)** is the most widely used index for measuring vegetation health from satellite imagery. It exploits a key property of plants: **healthy vegetation absorbs red light for photosynthesis but strongly reflects near-infrared (NIR) light**.

### The formula

```
NDVI = (NIR - Red) / (NIR + Red)
```

- **NIR** = Near-Infrared band reflectance
- **Red** = Red band reflectance

### Why it works

| Surface | Red Light | NIR Light | NDVI |
|---------|-----------|-----------|------|
| Healthy vegetation | Low (absorbed) | High (reflected) | **0.3 to 0.9** |
| Stressed vegetation | Medium | Medium | **0.1 to 0.3** |
| Bare soil | Similar | Similar | **0.0 to 0.1** |
| Water | Low | Very low | **-0.3 to 0.0** |
| Clouds/Snow | High | High | **~0** |

### Interpreting NDVI values

- **0.6 - 1.0**: Dense, healthy vegetation (forests, crops at peak growth)
- **0.3 - 0.6**: Moderate vegetation (shrubs, grassland, crops)
- **0.1 - 0.3**: Sparse vegetation or stressed plants
- **-0.1 - 0.1**: Bare soil, rock, sand, urban areas
- **-1.0 - -0.1**: Water, snow, clouds

### Which bands to use?

| Satellite | Red Band | NIR Band |
|-----------|----------|----------|
| **Sentinel-2** | Band 4 (index 3) | Band 8 (index 7) |
| **Landsat 8/9** | Band 4 (index 3) | Band 5 (index 4) |
| **Landsat 5/7** | Band 3 (index 2) | Band 4 (index 3) |

**Note:** Python uses 0-based indexing, so "Band 4" is accessed as index 3.

### Further reading

- [USGS: What is NDVI?](https://www.usgs.gov/special-topics/remote-sensing-phenology/science/ndvi-foundation-remote-sensing-phenology)
- [NASA Earth Observatory: Measuring Vegetation](https://earthobservatory.nasa.gov/features/MeasuringVegetation)
- [Sentinel Hub: NDVI](https://custom-scripts.sentinel-hub.com/sentinel-2/ndvi/)

In [ ]:
def calculate_ndvi(raster, red_band=3, nir_band=7):
    """
    Calculate NDVI from a multi-band raster.
    
    Parameters:
    - raster: numpy array with shape (bands, height, width)
    - red_band: index of red band (default 3 for Sentinel-2 Band 4)
    - nir_band: index of NIR band (default 7 for Sentinel-2 Band 8)
    
    Returns:
    - ndvi: 2D array with values from -1 to 1
    """
    # Extract bands and convert to float
    red = raster[red_band].astype(float)
    nir = raster[nir_band].astype(float)
    
    # Avoid division by zero
    np.seterr(divide='ignore', invalid='ignore')
    
    # Calculate NDVI
    ndvi = (nir - red) / (nir + red)
    
    # Set invalid values to NaN
    ndvi[~np.isfinite(ndvi)] = np.nan
    
    return ndvi

# Calculate NDVI for both dates
# Adjust band indices if using different satellite data!
ndvi_before = calculate_ndvi(before, red_band=3, nir_band=7)
ndvi_after = calculate_ndvi(after, red_band=3, nir_band=7)

print("NDVI Before:")
print(f"  Min:  {np.nanmin(ndvi_before):.3f}")
print(f"  Max:  {np.nanmax(ndvi_before):.3f}")
print(f"  Mean: {np.nanmean(ndvi_before):.3f}")

print("\nNDVI After:")
print(f"  Min:  {np.nanmin(ndvi_after):.3f}")
print(f"  Max:  {np.nanmax(ndvi_after):.3f}")
print(f"  Mean: {np.nanmean(ndvi_after):.3f}")

In [ ]:
# Visualize NDVI for both dates
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# NDVI Before
im1 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI - Before")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], shrink=0.8, label="NDVI")

# NDVI After
im2 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI - After")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], shrink=0.8, label="NDVI")

plt.suptitle("NDVI: Green = healthy vegetation, Red/Yellow = bare/stressed", y=1.02)
plt.tight_layout()
plt.show()

---

## Step 6: Calculate NDVI change

Now we can detect vegetation change by comparing NDVI between the two dates.

**Positive change** = vegetation increased (greening, regrowth)  
**Negative change** = vegetation decreased (clearing, drought, fire)

In [ ]:
# Calculate NDVI change (after minus before)
ndvi_change = ndvi_after - ndvi_before

print("NDVI Change statistics:")
print(f"  Mean:  {np.nanmean(ndvi_change):.3f}")
print(f"  Min:   {np.nanmin(ndvi_change):.3f}")
print(f"  Max:   {np.nanmax(ndvi_change):.3f}")

# Count pixels with significant change
significant_decrease = np.sum(ndvi_change < -0.1)
significant_increase = np.sum(ndvi_change > 0.1)
total_pixels = np.sum(~np.isnan(ndvi_change))

print(f"\nPixels with significant vegetation loss (< -0.1):  {significant_decrease:,} ({100*significant_decrease/total_pixels:.1f}%)")
print(f"Pixels with significant vegetation gain (> +0.1):  {significant_increase:,} ({100*significant_increase/total_pixels:.1f}%)")

---

## Step 7: Visualize NDVI change

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# NDVI Before
im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI Before")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# NDVI After
im1 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI After")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# NDVI Change
im2 = axes[2].imshow(ndvi_change, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
axes[2].set_title("NDVI Change\n(Green = vegetation gain, Red = vegetation loss)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], shrink=0.8, label="Change")

plt.tight_layout()
plt.show()

---

## Step 8: Zonal statistics

Summarize NDVI change values by zone (like QGIS Zonal Statistics tool). This tells you which areas experienced the most vegetation change.

In [ ]:
from rasterstats import zonal_stats

# Load zones
zones = gpd.read_file(RAW / "zones.geojson")

# Calculate zonal statistics on NDVI change
stats = zonal_stats(
    zones, 
    ndvi_change, 
    affine=before_meta["transform"],
    stats=["mean", "min", "max"],
    nodata=np.nan
)

# Add to GeoDataFrame
zones["ndvi_change_mean"] = [s["mean"] for s in stats]
zones["ndvi_change_min"] = [s["min"] for s in stats]
zones["ndvi_change_max"] = [s["max"] for s in stats]

zones[["name", "ndvi_change_mean", "ndvi_change_min", "ndvi_change_max"]].head()

---

## Step 9: Map zonal results

Visualize which zones experienced the most vegetation change.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

zones.plot(
    column="ndvi_change_mean",
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "Mean NDVI Change"},
    ax=ax
)

ax.set_title("Mean NDVI Change by Zone\n(Green = vegetation gain, Red = vegetation loss)")
ax.set_axis_off()
plt.show()

---

## Step 10: Export results

Save the zones with NDVI change statistics to the processed folder.

In [ ]:
# Save zones with change statistics to processed folder
zones.to_file(PROCESSED / "zones_change.gpkg", driver="GPKG")

print(f"Saved to: {PROCESSED / 'zones_change.gpkg'}")
print(f"\nYou can now open this file in QGIS from your processed folder!")

---

## Done!

You've completed NDVI-based vegetation change detection in Python:

1. Loaded and clipped satellite imagery to your study area
2. Calculated NDVI for both dates (measuring vegetation health)
3. Detected change by comparing NDVI values
4. Summarized NDVI change by administrative zones
5. Exported results for further analysis in QGIS

**Key concepts:**
- NDVI uses the ratio of NIR to Red light to measure vegetation
- Healthy plants reflect NIR strongly (high NDVI)
- Comparing NDVI over time reveals vegetation loss or gain

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`